In [ ]:
from typing_extensions import TypedDict, Literal
from typing import List
from langgraph.types import Command
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model("openai:gpt-4o")

dumb_llm = init_chat_model("openai:gpt-3.5-turbo")
average_llm = init_chat_model("openai:gpt-4")
smart_llm = init_chat_model("openai:gpt-5")

In [17]:
class State(TypedDict):

    question: str
    difficulty: str
    answer: str
    model_used: str

class DifficultyResponse(BaseModel):
    difficulty_level: Literal["easy", "medium", "hard"]


In [24]:
def dumb_node(state: State):
    response = dumb_llm.invoke(state["question"])
    return{
        "answer": response.content,
        "model_used": "gpt-3.5"
    }


def average_node(state: State):
    response = average_llm.invoke(state["question"])
    return{
        "answer": response.content,
        "model_used": "gpt-4o"
    }


def smart_node(state: State):
    response = smart_llm.invoke(state["question"])
    return{
        "answer": response.content,
        "model_used": "gpt-o3"
    }
def assess_difficulty(state: State):
    structured_llm = llm.with_structured_output(DifficultyResponse)

    response = structured_llm.invoke(
        f"""
        Assess the difficulty of this question
        Question: {state["question"]}

        -EASY: Simple facts, basic definition, yes/no answers
        -MEDIUM: Requires explanation, comparision, analysis
        -HARD: Complex reasoning, multiple stpes, deep experties. 
        """
    )

    difficulty_level = response.difficulty_level
    if difficulty_level == "easy":
        goto = "dumb_node"
    elif difficulty_level == "medium":
        goto = "average_node"
    elif difficulty_level == "hard":
        goto = "smart_node"

    return Command(
        goto = goto,
        update={
            "difficulty": difficulty_level,
        }
    )

In [25]:
graph_builder = StateGraph(State)

graph_builder.add_node("dumb_node", dumb_node)
graph_builder.add_node("average_node", average_node)
graph_builder.add_node("smart_node", smart_node)
graph_builder.add_node("assess_difficulty", assess_difficulty, destinations=("dumb_node","average_node","smart_node"))

graph_builder.add_edge(START, "assess_difficulty")

graph_builder.add_edge("dumb_node", END)
graph_builder.add_edge("average_node", END)
graph_builder.add_edge("smart_node", END)

graph = graph_builder.compile()

In [26]:
graph.invoke({"question": "Capital of Poland"})

/Users/bitnoorilee/projects/AiAgents/workflow-architectures/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=DifficultyResponse(difficulty_level='easy'), input_type=DifficultyResponse])
  return self.__pydantic_serializer__.to_python(


{'question': 'Capital of Poland',
 'difficulty': 'easy',
 'answer': 'Warsaw',
 'model_used': 'gpt-3.5'}